In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [2]:
def display_missing_data_info(dataframe, ascending=False):
    missing_values_count = dataframe.isnull().sum()
    missing_values_percentage = (dataframe.isnull().mean() * 100)
    missing_data = pd.DataFrame({
        'Missing Values': missing_values_count,
        'Percentage (%)': missing_values_percentage
    })
    
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    sorted_missing_data = missing_data.sort_values(by='Missing Values', ascending=ascending)

    print(sorted_missing_data)
    return sorted_missing_data
    

In [3]:
installment = pd.read_csv('../data/dseb63_installments_payments.csv')
y_train = pd.read_csv('../data/dseb63_application_train.csv')

In [4]:
installment = installment.merge(y_train[['SK_ID_CURR', 'TARGET']], on='SK_ID_CURR', how='left')


In [5]:
installment['SK_ID_PREV'].nunique()

549020

In [6]:
installment.shape

(7744758, 9)

In [7]:
installment_sample = installment[installment['SK_ID_CURR'] == 246424]
installment_sample

,SK_ID_PREV,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,SK_ID_CURR,TARGET
3249786,1776290,1.0,13,-631.0,-635.0,33643.845,33643.845,246424.0,0.0
3249787,1295333,1.0,11,-2084.0,-2089.0,9166.500,9166.500,246424.0,0.0
3249788,1776290,1.0,31,-91.0,-91.0,33643.845,33643.845,246424.0,0.0
3249789,1776290,1.0,32,-61.0,-61.0,33643.845,33643.845,246424.0,0.0
3249790,1776290,1.0,27,-211.0,-211.0,33643.845,33643.845,246424.0,0.0
...,...,...,...,...,...,...,...,...,...
3250073,2244271,1.0,1,-556.0,-557.0,16639.875,16639.875,246424.0,0.0
3250074,1570060,1.0,2,-1579.0,-1581.0,8679.870,8679.870,246424.0,0.0
3250075,1295333,1.0,18,-1874.0,-1875.0,9005.400,9005.400,246424.0,0.0
3250076,2535912,1.0,28,-495.0,-496.0,39915.540,39915.540,246424.0,0.0


In [8]:
installment_sample.sort_values(by=['SK_ID_PREV', 'NUM_INSTALMENT_NUMBER'], ascending=False, inplace=True)
# installment_185737.sort_values(by=, ascending=False, inplace=True)
installment_sample.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_20324\1388967542.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  installment_sample.sort_values(by=['SK_ID_PREV', 'NUM_INSTALMENT_NUMBER'], ascending=False, inplace=True)


,SK_ID_PREV,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,SK_ID_CURR,TARGET
3249939,2605449,1.0,6,-616.0,-635.0,18017.145,18017.145,246424.0,0.0
3249796,2605449,1.0,5,-646.0,-646.0,18091.035,18091.035,246424.0,0.0
3250002,2605449,1.0,4,-676.0,-677.0,18091.035,18091.035,246424.0,0.0
3249962,2605449,1.0,3,-706.0,-752.0,18091.035,18091.035,246424.0,0.0
3250059,2605449,1.0,2,-736.0,-752.0,18091.035,18091.035,246424.0,0.0


## Feature for installments payment
- Group by id prev 
- Count num install
- version: max
- Late: mean
- Early: mean
- Last payment date
- Total money

-Groupby idprev + install num
- Mean value each payment

## Group by id curr
- Count install
- Sum money
- Mean early
- Mean late
- Mean value each payment
- Count last payment 3-6-9-12-24

### Sep by duration (less than 12 and more than 12) (ngan han / trung han)
- Count install 
- Sum money
- Mean early
- Mean late
- Mean value each payment
- Variance each payment
- Count payment 3-12-24
- Sum payment 3-12-24

In [ ]:
installment['DIFF'] = installment['DAYS_INSTALMENT'] - installment['DAYS_ENTRY_PAYMENT']
installment['LATE'] = installment['DIFF'] > 0
installment['EARLY'] = installment['DIFF'] <= 0

installment['LATE'] *= installment['DIFF']
installment['EARLY'] *= -installment['DIFF']

installment["LAST_30_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -30
installment["LAST_60_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -60
installment["LAST_90_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -90
installment["LAST_180_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -180
installment["LAST_365_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -365
installment["LAST_730_DAYS"] = installment["DAYS_ENTRY_PAYMENT"] >= -730

installment["PAYMENT_LAST_30_DAYS"] = installment["LAST_30_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_60_DAYS"] = installment["LAST_60_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_90_DAYS"] = installment["LAST_90_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_180_DAYS"] = installment["LAST_180_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_365_DAYS"] = installment["LAST_365_DAYS"] * installment["AMT_PAYMENT"]
installment["PAYMENT_LAST_730_DAYS"] = installment["LAST_730_DAYS"] * installment["AMT_PAYMENT"]

In [14]:
installment['COUNT'] = 1

In [ ]:
gb_installment = installment.groupby('SK_ID_PREV').agg({'SK_ID_CURR':'first', 'COUNT':'sum', 'LATE':'mean', 'EARLY':'mean', 'DAYS_ENTRY_PAYMENT':'max', 'AMT_PAYMENT':'sum', 'TARGET':'first', 'LAST_30_DAYS':'sum', 'LAST_60_DAYS':'sum', 'LAST_90_DAYS':'sum', 'LAST_180_DAYS':'sum', 'LAST_365_DAYS':'sum', 'LAST_730_DAYS':'sum'}).reset_index()
gb_installment.rename(columns={'COUNT':'NUM_INSTALMENT', 'LATE':'LATE_PAYMENT', 'EARLY':'EARLY_PAYMENT', 'DAYS_ENTRY_PAYMENT':'LASTEST_PAYMENT_DATE', 'AMT_PAYMENT':'TOTAL_PAYMENT'}, inplace=True)
gb_installment.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT,LATE_PAYMENT,EARLY_PAYMENT,LASTEST_PAYMENT_DATE,TOTAL_PAYMENT,TARGET
0,1000001,117953.0,2,16.000000,0.000000,-244.0,68443.425,0.0
1,1000003,6707.0,3,15.333333,0.000000,-49.0,14854.050,0.0
2,1000004,187620.0,7,26.714286,0.000000,-695.0,33523.155,NaN
3,1000005,83759.0,11,8.818182,0.363636,-1433.0,147021.705,0.0
4,1000007,25033.0,5,16.800000,0.000000,-10.0,56234.025,NaN


In [16]:
gb_installment

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT,LATE_PAYMENT,EARLY_PAYMENT,LASTEST_PAYMENT_DATE,TOTAL_PAYMENT,TARGET
0,1000001,117953.0,2,16.000000,0.000000,-244.0,68443.425,0.0
1,1000003,6707.0,3,15.333333,0.000000,-49.0,14854.050,0.0
2,1000004,187620.0,7,26.714286,0.000000,-695.0,33523.155,NaN
3,1000005,83759.0,11,8.818182,0.363636,-1433.0,147021.705,0.0
4,1000007,25033.0,5,16.800000,0.000000,-10.0,56234.025,NaN
...,...,...,...,...,...,...,...,...
549015,2843490,158969.0,4,0.000000,2.750000,-2616.0,16157.745,0.0
549016,2843491,277329.0,10,14.200000,0.000000,-35.0,254219.850,0.0
549017,2843492,36159.0,12,6.250000,0.000000,-344.0,262990.800,NaN
549018,2843494,70155.0,2,10.000000,0.000000,-770.0,982794.465,0.0
